<a href="https://colab.research.google.com/github/munawirmt/Plant_Disease_App/blob/main/plant_disease_diagnosis_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Plant Disease Diagnostics + Smart Crop Advisor (End-to-End AI SaaS App)**

**Step 1: Environment Setup & Kaggle Dataset Import**

In [1]:
## install libreries

import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

In [2]:
## conforming availability of GPU
print("GPU Available: ", tf.config.list_physical_devices('GPU'))

GPU Available:  [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
## importing kaggle datahub
!pip install kagglehub
import kagglehub

In [8]:
import os
import tensorflow as tf


# Direct Dataset Download via wget (Official Repository Zip)
!wget -q https://github.com/spMohanty/PlantVillage-Dataset/archive/refs/heads/master.zip -O plantvillage.zip

print("Unzipping dataset...")
!unzip -q plantvillage.zip

# Path setting
path = "/content/PlantVillage-Dataset-master/raw/color"
print("Dataset successfully downloaded and extracted to path:", path)

Unzipping dataset...
Dataset successfully downloaded and extracted to path: /content/PlantVillage-Dataset-master/raw/color


**Step 2: Data Preprocessing & Augmentation**

In [14]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 64

datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_generator = datagen.flow_from_directory(
    path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)


Found 43456 images belonging to 38 classes.
Found 10849 images belonging to 38 classes.


In [10]:
# Class Labels to use in the time of deployment
class_indices = train_generator.class_indices
labels = {v: k for k, v in class_indices.items()}
print("Classes identified:", len(labels))

Classes identified: 38


**Step 3: EfficientNetB0 (Transfer Learning) Model Training**

In [11]:
## transfer learning model building
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # Base layers freezing

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(256, activation='relu')(x)
predictions = Dense(len(labels), activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [12]:
## callbacks
checkpoint = ModelCheckpoint('plant_disease_model.h5', monitor='val_accuracy', save_best_only=True, mode='max')
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

In [16]:
# Speed-Up Optimized Training (Keras 3 Compatible)
history = model.fit(
    train_generator,
    epochs=5,
    validation_data=val_generator,
    callbacks=[checkpoint, early_stop],
    steps_per_epoch=200,
    validation_steps=50
)

Epoch 1/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 841ms/step - accuracy: 0.0978 - loss: 3.3520

200/200 ━━━━━━━━━━━━━━━━━━━━ 241s 1s/step - accuracy: 0.0973 - loss: 3.3561 - val_accuracy: 0.0978 - val_loss: 3.3573
Epoch 2/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 860ms/step - accuracy: 0.0996 - loss: 3.3635

200/200 ━━━━━━━━━━━━━━━━━━━━ 216s 1s/step - accuracy: 0.0995 - loss: 3.3572 - val_accuracy: 0.1006 - val_loss: 3.3623
Epoch 3/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 217s 1s/step - accuracy: 0.1009 - loss: 3.3611 - val_accuracy: 0.0950 - val_loss: 3.3648
Epoch 4/5
 79/200 ━━━━━━━━━━━━━━━━━━━━ 1:43 852ms/step - accuracy: 0.1017 - loss: 3.3507

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


200/200 ━━━━━━━━━━━━━━━━━━━━ 111s 556ms/step - accuracy: 0.1023 - loss: 3.3614 - val_accuracy: 0.1022 - val_loss: 3.3329
Epoch 5/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 221s 1s/step - accuracy: 0.1009 - loss: 3.3496 - val_accuracy: 0.1019 - val_loss: 3.3569


**Step 4: Trained Model & Class Labels Save**

In [17]:
import json
from google.colab import files

In [18]:
# save class Indices
with open('class_indices.json', 'w') as f:
    json.dump(labels, f)

In [19]:
# download files
files.download('plant_disease_model.h5')
files.download('class_indices.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>